In [2]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY: ")

if not os.getenv("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter TAVILY_API_KEY: ")

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Enter PINECONE_API_KEY: ")

print("GROQ_API_KEY configured:", bool(os.getenv("GROQ_API_KEY")))
print("TAVILY_API_KEY configured:", bool(os.getenv("TAVILY_API_KEY")))
print("PINECONE_API_KEY configured:", bool(os.getenv("PINECONE_API_KEY")))

GROQ_API_KEY configured: True
TAVILY_API_KEY configured: True
PINECONE_API_KEY configured: True


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

In [3]:
from langchain_community.document_loaders import WebBaseLoader

SOURCE_URL = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

loader = WebBaseLoader(
    web_paths=(SOURCE_URL,),
    requests_kwargs={
        "headers": {
            "User-Agent": "Mozilla/5.0 Agentic-RAG-Industry-Demo"
        }
    },
)

raw_docs = loader.load()

print("Loaded documents:", len(raw_docs))
print("Source:", raw_docs[0].metadata.get("source"))
print("\nPreview:\n")
print(raw_docs[0].page_content[:1500])

C:\Users\nathm\AppData\Local\Temp\ipykernel_22272\625167974.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded documents: 1
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag

Preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextGraph APIFunctional APIAdditional resourcesUse docs programmaticallyLangChain AcademyCase studiesGet helpOn this pageConceptsSetupSet up LangSmithPreprocess documentsCrea

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True,
)

chunks = splitter.split_documents(raw_docs)

print("Total chunks:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0].page_content[:900])

Total chunks: 28

First chunk preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextGraph APIFunctional APIAdditional resourcesUse docs programmaticallyLangChain AcademyCase studiesGet helpOn this pageConceptsSetupSet up LangSmithPreprocess documentsCre


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(
    model_name='sentence_transformers/All-MiniLm-L6-v2',
    model_kwargs={"normalize_embeddings":True}

)
sample_vector=embeddings.embed_query("Hi Manish!")
print("Embedding dimensions:", len(sample_vector))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2493.40it/s]


Embedding dimensions: 384


In [ ]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time

INDEX_NAME = "industry-agentic-rag-kb"
NAMESPACE = "langgraph-agentic-rag"

# Connect to Pinecone
pc=Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
# Create the index only if it does not already exist.
indexes=[index_info['name'] for index_info in pc.list_indexes() ]
if INDEX_NAME not in indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric='cosine',
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1',
        ),
        
    )
    # Wait until Pinecone reports the new index as ready.
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

print("Pinecone index ready:", INDEX_NAME)

# Upload the document chunks and create the LangChain vector store.
vectorstore=PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE,
    }
)

print("Pinecone vector database and retriever are ready.")


Pinecone index ready: industry-agentic-rag-kb
Pinecone vector database and retriever are ready.


In [ ]:
# #LOAD EXISITING INDEX

# INDEX_NAME = "industry-agentic-rag-kb"
# NAMESPACE = "langgraph-agentic-rag"
# pc=Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
# index=pc.Index(INDEX_NAME)

# vectorstore=PineconeVectorStore(
#     index=index,
#     embedding=embeddings,
#     namespace=NAMESPACE
# )

# retriever=vectorstore.as_retriever(
#     search_kwargs={"k":4,
#                    "namespace":NAMESPACE},
#     search_type='similarity'
# )




In [ ]:
test_question="what is langchain"
answer=retriever.invoke(test_question)
print(answer)

In [ ]:
from langchain_tavily import TavilySearch

search_web=TavilySearch(
    max_result=3,
    topic='general',
    include_answer=True,
    include_raw_content=True,
    include_images=True,
    include_image_descriptions=True,
    search_depth="basic",
    time_range="day",
    include_domains=None,
)
print("Tavily Search is ready!")

In [ ]:
from typing import Optional,Literal,f
from pydantic import BaseModel,Field
from typing_extensions import TypedDict
from langchain_core.documents import Document

class RouteDecision(BaseModel):
    route:Literal['kb','direct']=Field(
        description="Use kb for agentic rag needing docs, otherwise use direct for simple chat"
    )

class EvidenceGrade(BaseModel):
    grade:Literal["good","bad"]=Field(
        description="if evidence is good it can be used for answer if evidence is bad it cannot be used for generating the answer"
    )

class AgenticState(TypedDict):
    question:str
    current_query:str
    kb_docs:list[Document]
    web_result:str
    kb_grade:str
    web_grade:str
    answer:str
    source_used:str
    retry_count:int
    


In [ ]:
router_llm=llm.with_structured_output(RouteDecision,method='json_mode')
def route_state(state:AgenticState):
    question=state['question']
    decision=router_llm.invoke(f"""
    You are a router for an Agentic RAG assistant.

Route to "kb" if the user asks about:
- Agentic RAG
- LangGraph Agentic RAG workflow
- retrieval grading
- query rewriting
- RAG architecture
- retriever tools
- web fallback in RAG

Route to "direct" only for greetings, thanks, or very simple conversation.

Question:
{question}

Return your response as valid JSON.
Example:
{{"route": "kb"}}
""")
    print('[Router]',decision.route)
    return {
        "question":question,
        "route":decision.route
    }
def route_after_router(state:AgenticState)->Literal['retrieve_kb','direct_answer']:
    if state['source_used']=='kb':
        return 'retreive_kb'
    return 'direct_answer'

In [ ]:
def retrieve_kb(state:AgenticState):
    query=state['question']
    docs=retriever.invoke(query)
    print(f"[KB Retriever] Query: {query}")
    print(f"[KB Retriever] Retrieved: {len(docs)} chunks")

    return {"kb_docs": docs}

In [ ]:
kb_grader_llm=llm.with_structured_output(EvidenceGrade,method='json_mode')

def kb_grader(state:AgenticState):
    question=state['question']
    context='\n\n'.join(
        f"source:{doc.metadata.get('source')}\n{doc.page_content}"
        for doc in state['kb_docs']
    )
    grade=kb_grader_llm.invoke("""
                               You are an evidence grader.

Question:
{question}

Private KB evidence:
{context}

Can this private KB evidence answer the question?
Return "good" if it can answer.
Return "weak" if it cannot answer or is incomplete.

Return your response as valid JSON.
Example:
{{"grade": "good"}}
""")
    print(grade.grade)
    return {"kb_grade",grade.grade}

def grade_after_grader(state:AgenticState)->Literal['good','bad']:
    if state['kb_grade']=='good':
        return "generate_from_kb"
    return "search_web"
    


In [ ]:
def search_web(state:AgenticState):
    query=state['question']
    print(f"Tavily search query:{query}")
    web_search=TavilySearch.invoke(query)
    if isinstance(web_search,dict):
        result=web_search.get('results',"")
        ans=web_search.get("answer","")
        lines=[]
        if ans:
            lines.append(ans)
        for item in result:
            title=item.get("title","")
            url=item.get("url","")
            content=item.get("content","")
            lines.append(f"Title: {title}\nURL: {url}\nContent: {content}")
        web_text = "\n\n".join(lines) if lines else str(result)
    else:
        web_text = str(result)
        print("[Tavily Search] Result characters:", len(web_text))

    return {
        "web_results": web_text,
        "source_used": "web",
    }

In [ ]:
web_grader_llm=llm.with_stuctured_output(EvidenceGrade,method='json_mode')
def web_grader(state:AgenticState):
    question=state['question']
    web_result=state['web_result']
    grader=web_grader_llm.invoke('''
                                 You are an evidence grader.

Question:
{question}

Web search evidence:
{web_results}

Can this web evidence answer the question?
Return "good" if it can answer.
Return "weak" if it cannot answer or is incomplete.

Return your response as valid JSON.
Example:
{{"grade": "good"}}
  ''')
    print('[WEB_GARDE]',grader.grade)
    return {
        "grade":grader.grade
    }
    
MAX_RETRIES=1
def decide_after_web_grade(state:AgenticState)->Literal["generate_from_web","rewrite_query","'insuffiecient_answer"]:
    if state['web_grade']=="good":
        return "generate_from_web"
    elif MAX_RETRIES>state["retry_count"]:
        return "rewrite_query"
    else:
        return "insuffiecient_answer"

In [ ]:
from langchain_core.messages import HumanMessage
def rewrite_query(state:AgenticState):
    question=state['question']
    retry_count=state['retry_count']+1
    rewritten = llm.invoke(f'''
Rewrite the question for better retrieval and web search.

Rules:
- Preserve original intent.
- Make it specific and search-friendly.
- Do not answer.
- Return only the rewritten query.

Original question:
{question}
''').content.strip()

    print("[Rewriter]", rewritten)

    return {
        "current_query": rewritten,
        "retry_count": retry_count,
    }
    

In [ ]:
def generate_from_kb(state:AgenticState):
    question=state['question']
    context="\n\n".join(f"[source{doc.metadata.get('source')}]\n{doc.page_content}"
    for doc in state['kb_docs'])
    ans=llm.invoke(
                  ''' Answer using ONLY the private KB context.

Rules:
- Beginner-friendly explanation.
- Do not invent unsupported details.
- Mention that the answer is based on the private KB.
- Include source type: Private KB.

Question:
{question}

Private KB context:
{context}
''').content

    return {
        "answer": answer,
        "source_used": "private_kb",
    }

In [ ]:
def generate_from_web(state: AgenticState):
    question = state["question"]
    web_context = state["web_results"]

    ans= llm.invoke(f'''
You are a technical instructor.

The private KB was insufficient, so web search was used.

Answer using ONLY the web search context.

Rules:
- Beginner-friendly explanation.
- Do not invent unsupported details.
- Mention that the answer is based on Tavily web search.
- Include source type: Web Search.
- If URLs are present in context, include the most useful URLs.

Question:
{question}

Web search context:
{web_context}
''').content

    return {
        "answer": answer,
        "source_used": "web_search",
    }

In [ ]:
def direct_answer(state: AgenticState):
    question = state["question"]

    answer = llm.invoke(f'''
Respond briefly and naturally.

Message:
{question}
''').content

    return {
        "answer": answer,
        "source_used": "direct",
    }

In [ ]:
def answer_insufficient(state: AgenticState):
    answer = (
        "I could not find enough reliable evidence in the private knowledge base "
        "or the web search results to answer this confidently. "
        "Please provide more specific documents or rephrase the question."
    )

    return {
        "answer": answer,
        "source_used": "insufficient_evidence",
    }

In [ ]:
from langgraph.graph import  END,START,StateGraph
workflow=StateGraph(AgenticState)
workflow.add_node("route_question",route_state)
workflow.add_node('retreive_from_kb',retrieve_kb)
workflow.add_node('grade_kb',kb_grader_llm)
workflow.add_node('search_web',search_web)
workflow.add_node("web_grader",web_grader)
workflow.add_node("rewrite",rewrite_query)
workflow.add_node("generate_from_kb",generate_from_kb)
workflow.add_node("generate_from_search",generate_from_web)
workflow.add_node("direct_answer",direct_answer)
workflow.add_node("insufficient_answer",answer_insufficient)
workflow.add_edge(START,'route-question')
workflow.add_conditional_edges(
    'route_question',
    route_after_router,{
    "retrieve_kb":"retrieve_kb",
    "direct_answer":"direct_answer"
    },
)
workflow.add_edge("retrieve_kb","grade_kb")
workflow.add_conditional_edges(
    'grade_kb',
    grade_after_grader,{
        "generate_from_kb":"generate_from_kb",
        "search_web":"search_Web",
    },   
)
workflow.add_edge("search_web","web_grader")
workflow.add_conditional_edges(
    "web_grader",
    decide_after_web_grade,{
         "generate_from_web": "generate_from_web",
        "rewrite_query": "rewrite_query",
        "answer_insufficient": "answer_insufficient",
    },
)
workflow.add_edge("rewrite_query", "retrieve_kb")
workflow.add_edge("retrieve_kb",END)
workflow.add_edge("generate_from_search",END)
workflow.add_edge("direct_answer", END)
workflow.add_edge("answer_insufficient",END)
graph=workflow.compile()



In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph visualization is optional. The graph can still run.")
    print("Reason:", e)